# Triage Module Evaluation

This notebook evaluates the `TriageModule` (Linguistic Normalizer and Language Detector) using a custom dataset. It fills in the `Detected Language` and `Normalized Output` columns for each entry in the evaluation dataset.

In [2]:
# 0. Install required libraries
%pip install pandas openpyxl tqdm requests python-dotenv rich

You should consider upgrading via the '/Users/sebastianrafaellachica/codingprojects/Legal-Adaptive-Routing-Framework/myvenv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys
import os
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# 1. Add project root to sys.path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.config import FrameworkConfig

# 2. Load environment variables from project root .env
load_dotenv(os.path.join(project_root, '.env'))

print("Environment setup complete.")

/Users/sebastianrafaellachica/codingprojects/Legal-Adaptive-Routing-Framework/myvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment setup complete.


In [ ]:
# ------------------------------------------------------------------
# CONFIGURATION: Manually paste your OpenRouter API key here
# ------------------------------------------------------------------
OPENROUTER_API_KEY = "" # <--- PASTE API KEY HERE
# ------------------------------------------------------------------

if not OPENROUTER_API_KEY:
    print("No manual API key provided. Falling back to .env or environment variables.")
    api_key = os.getenv("OPENROUTER_API_KEY")
else:
    print("Using manually provided API key.")
    api_key = OPENROUTER_API_KEY

if not api_key:
    print("WARNING: No API key found. Module initialization might fail.")

Using manually provided API key.


In [11]:
# Initialize Triage Module
triage = TriageModule(api_key=api_key)

print(f"Triage Module initialized using model: {FrameworkConfig._TRIAGE_MODEL}")

Triage Module initialized using model: qwen/qwen-turbo


In [12]:
# Path to the evaluation dataset
dataset_path = 'dataset/Triage-Module-Evaluation-Dataset.xlsx'

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

# Load the dataset
df = pd.read_excel(dataset_path)

print(f"Loaded dataset with {len(df)} rows.")
print("Columns:", df.columns.tolist())

# Identify the input column (change this matches your Excel file)
input_column = "Prompt" 
if input_column not in df.columns:
    # Attempting to find common names if 'Prompt' is not present
    candidates = ["Input", "Query", "Prompt", "Text", "raw_text"]
    for cand in candidates:
        if cand in df.columns:
            input_column = cand
            print(f"Using '{input_column}' as the input column.")
            break

df.head()

Loaded dataset with 149 rows.
Columns: ['prompts', 'language', 'llm_normalized_output', 'llm_detected_language']


,prompts,language,llm_normalized_output,llm_detected_language
0,"Attorney, hawak po ng employer ko ang passport...",Tagalog,NaN,NaN
1,Hindi ako pinayagan mag-day off ng amo ko for ...,Tagalog,NaN,NaN
2,May karapatan ba akong magreklamo kapag sinasa...,Tagalog,NaN,NaN
3,Gusto ko na po umuwi pero ayaw ibigay ang tick...,Tagalog,NaN,NaN
4,Pinapirma ako ng kontrata na iba sa pinag-usap...,Tagalog,NaN,NaN


In [ ]:
# Target columns
lang_col = "Detected Language"
norm_col = "Normalized Output"

# Initialize columns if they don't exist
if lang_col not in df.columns:
    df[lang_col] = None
if norm_col not in df.columns:
    df[norm_col] = None

print("Starting evaluation processing...")

for index, row in tqdm(df.iterrows(), total=len(df)):
    raw_input = row[input_column]
    
    if pd.isna(raw_input) or str(raw_input).strip() == "":
        continue
        
    try:
        # Execute triage processing
        result = triage._process_request_(str(raw_input))
        
        # Update the dataframe
        df.at[index, lang_col] = result.get("detected_language")
        df.at[index, norm_col] = result.get("normalized_text")
        
    except Exception as e:
        print(f"Error processing row {index}: {str(e)}")
        df.at[index, lang_col] = "ERROR"
        df.at[index, norm_col] = str(e)

print("Processing complete.")

In [ ]:
# Save the updated dataset
output_path = 'dataset/Triage-Module-Evaluation-Results.xlsx'
df.to_excel(output_path, index=False)

print(f"Results saved to: {output_path}")
df.head()